In [1]:
# Importar librerías
import pandas as pd
from src.config import data_folder, tickers_file

%load_ext autoreload
%autoreload 2
from src.extract import *

In [2]:
df_simfin_unfiltered = inicializar_datos_simfin(update_tickers=True)
df_simfin_unfiltered.shape

Generando universo de tickers...
Se extraen datos de simFin:
Dataset "us-income-quarterly" on disk (2 days old).
- Loading from disk ... Done!
Dataset "us-balance-quarterly" on disk (2 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" on disk (2 days old).
- Loading from disk ... Done!
Dataset "us-income-banks-quarterly" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-balance-banks-quarterly" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-banks-quarterly" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-income-insurance-quarterly" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-balance-insurance-quarterly" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-insurance-quarterly" on disk (1 days old).
- Loading from disk ... Done!
Fichero de tickers guardado exitosamente. Universo total: 549 tickers.


(50230, 82)

In [3]:
df_tickers_universe = pd.read_csv(tickers_file)
df_tickers_universe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 549 entries, 0 to 548
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Ticker  549 non-null    object
dtypes: object(1)
memory usage: 4.4+ KB


In [4]:
tickers_universe_list = df_tickers_universe['Ticker'].tolist()
# Extraer precios de los tickers y del índice SPY
df_prices = extraer_precios(tickers_universe_list)

# Se extrae precio del Índice de Mercado para usar en cálculos y se guarda en fichero (demora unos minutos)
df_index = extraer_precios(['SPY'])
df_index.to_parquet(f"{data_folder}/market_index.parquet")

df_prices.info()

Iniciando descarga masiva para 549 tickers únicos...


[*********************100%***********************]  549 of 549 completed

27 Failed downloads:
['QRTEA', 'SAVE', 'FLT', 'WBA', 'HES', 'IPG', 'MDC', 'TPX', 'SPR', 'X', 'GMS', 'SKX', 'HOUS', 'CCH', 'TM-28', 'JNPR', 'HBI', 'APY', 'K', 'FI', 'FL', 'PDCO', 'SQ', 'DISH', 'RYI', 'JWN']: YFPricesMissingError('possibly delisted; no price data found  (period=6y) (Yahoo error = "No data found, symbol may be delisted")')
['QVC']: YFPricesMissingError('possibly delisted; no price data found  (period=6y)')


Reestructurando los datos...
Extracción completada.
Iniciando descarga masiva para 1 tickers únicos...


[*********************100%***********************]  1 of 1 completed

Reestructurando los datos...
Extracción completada.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13033 entries, 0 to 13032
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    13033 non-null  datetime64[ns]
 1   Ticker  13033 non-null  object        
 2   Close   13033 non-null  float64       
 3   Open    13033 non-null  float64       
 4   Volume  13033 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 509.2+ KB


In [5]:
# Actualizar el universo de tickers con los que se obtuvieron precios
tickers_universe_list = actualizar_universo_tickers(df_prices)
print("Cantidad de tickers actualizado:", len(tickers_universe_list))

Cantidad de tickers actualizado: 522


In [6]:
# Filtrar datos de simFin
df_simfin = df_simfin_unfiltered[df_simfin_unfiltered['Ticker'].isin(tickers_universe_list)]
df_simfin.shape

(9937, 82)

In [7]:
"""
Obtener lista de tickers del S&P 500 desde el fichero constituents.csv.
De dicho fichero se obtiene para los tickers que pertenecen al índice, 
la fecha en la que fueron agregados con la columna "DataAdded".
Si no existe el fichero, lo descarga desde GitHub.
Para actualizar la lista de componentes, cambiar force_update a True, ejecutar y volver a dejar en False
"""
ruta_sp500 = descargar_constituents_sp(force_update=False) 

# Cargar y limpiar datos
df_tickers_sp_raw = pd.read_csv(ruta_sp500)
df_tickers_sp = limpiar_constituents_sp(df_tickers_sp_raw)

df_tickers_sp.info()

Usando archivo constituents.csv local.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Ticker     503 non-null    object
 1   DateAdded  503 non-null    object
dtypes: object(2)
memory usage: 8.0+ KB


In [8]:
# Unir df_prices y df_tickers
df_merged = pd.merge(
    df_prices,
    df_tickers_sp,
    on= 'Ticker',
    how= 'left'
)
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13033 entries, 0 to 13032
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       13033 non-null  datetime64[ns]
 1   Ticker     13033 non-null  object        
 2   Close      13033 non-null  float64       
 3   Open       13033 non-null  float64       
 4   Volume     13033 non-null  float64       
 5   DateAdded  7572 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(2)
memory usage: 611.1+ KB


In [9]:
# Extraer info de sector e industria (demora unos minutos)
df_info = extraer_info(tickers_universe_list)
df_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522 entries, 0 to 521
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Ticker    522 non-null    object
 1   Sector    522 non-null    object
 2   Industry  522 non-null    object
dtypes: object(3)
memory usage: 12.4+ KB


In [10]:
# Se unen los dataframes
df_with_info = pd.merge(
    df_merged,
    df_info,
    on= 'Ticker',
    how= 'left'
)
df_with_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13033 entries, 0 to 13032
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       13033 non-null  datetime64[ns]
 1   Ticker     13033 non-null  object        
 2   Close      13033 non-null  float64       
 3   Open       13033 non-null  float64       
 4   Volume     13033 non-null  float64       
 5   DateAdded  7572 non-null   object        
 6   Sector     13033 non-null  object        
 7   Industry   13033 non-null  object        
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 814.7+ KB


In [11]:
# Extraer datos financieros de yfinance (últimos 4 reportes trimestrales)
"""
yfinance no incluye la fecha de publicación, sólo la fecha de los Estados Financieros

Para evitar el "LookAhead" bias, se ofrecen 2 alternativas con el parametro "aproximar_fechas":

aproximar_fechas = True: 
-- Se estima la fecha de publicación con una regla de 30 días por defecto (editable en src/config.py)
-- Demora de ejecución: Aproximadamente 10 minutos

aproximar_fechas = False: 
-- Obtiene la fecha real de publicación usando yf.earnings_dates
-- Se aplica la regla de 30 días sólo si no se encuentra
-- Demora de ejecución: puede demorar 30 minutos o más
"""

df_yfinance = extraer_financials(tickers_universe_list, aproximar_fechas = True)
df_yfinance.info()

Sin datos financieros trimestrales para CNLPL
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2084 entries, 0 to 2083
Data columns (total 22 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Date                           2084 non-null   datetime64[ns]
 1   Total Revenue                  2078 non-null   float64       
 2   Gross Profit                   1949 non-null   float64       
 3   Operating Income               1963 non-null   float64       
 4   Net Income                     2078 non-null   float64       
 5   EBITDA                         1963 non-null   float64       
 6   Basic Average Shares           2039 non-null   float64       
 7   Cash And Cash Equivalents      2073 non-null   float64       
 8   Current Debt                   1562 non-null   float64       
 9   Long Term Debt                 1954 non-null   float64       
 10  Total Debt                     2059 no

In [12]:
# Definir columnas a mantener en simFin para que coincidan y estandarizar antes de unir
cols_yfinance = df_yfinance.columns
df_simfin_clean = estandarizar_simfin(df_simfin, cols_yfinance)

df_financials_completo = unir_financials(df_yfinance, df_simfin_clean)
df_financials_completo.info()

Se han encontrado 0 filas con Ticker y Date solapados.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11643 entries, 0 to 11642
Data columns (total 22 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Ticker                         11643 non-null  object        
 1   Date                           11627 non-null  datetime64[ns]
 2   Total Revenue                  11623 non-null  float64       
 3   Gross Profit                   10636 non-null  float64       
 4   Operating Income               11508 non-null  float64       
 5   Net Income                     11623 non-null  float64       
 6   EBITDA                         11508 non-null  float64       
 7   Basic Average Shares           11576 non-null  float64       
 8   Cash And Cash Equivalents      11588 non-null  float64       
 9   Current Debt                   8845 non-null   float64       
 10  Long Term Debt             

In [13]:
# Unir dataframe de precios con datos financieros
df_final = pd.merge(
    df_with_info, 
    df_financials_completo, 
    on=['Date', 'Ticker'],
    how='left'
)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13033 entries, 0 to 13032
Data columns (total 28 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Date                           13033 non-null  datetime64[ns]
 1   Ticker                         13033 non-null  object        
 2   Close                          13033 non-null  float64       
 3   Open                           13033 non-null  float64       
 4   Volume                         13033 non-null  float64       
 5   DateAdded                      7572 non-null   object        
 6   Sector                         13033 non-null  object        
 7   Industry                       13033 non-null  object        
 8   Total Revenue                  11506 non-null  float64       
 9   Gross Profit                   10523 non-null  float64       
 10  Operating Income               11391 non-null  float64       
 11  Net Income     

In [14]:
# Limpieza final
df_final_clean = limpieza_final(df_final)
df_final_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11391 entries, 0 to 11390
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Date                         11391 non-null  datetime64[ns]
 1   Ticker                       11391 non-null  object        
 2   Close                        11391 non-null  float64       
 3   Open                         11391 non-null  float64       
 4   Volume                       11391 non-null  float64       
 5   DateAdded                    6705 non-null   object        
 6   Sector                       11391 non-null  object        
 7   Industry                     11391 non-null  object        
 8   TotalRevenue                 11391 non-null  float64       
 9   GrossProfit                  10523 non-null  float64       
 10  OperatingIncome              11391 non-null  float64       
 11  NetIncome                    11391 non-nu

In [15]:
# Guardar datos extraidos en fichero raw_data
df_final_clean.to_parquet(f"{data_folder}/raw_data.parquet")
print("Fichero guardado en la carpeta",data_folder)

Fichero guardado en la carpeta data
